In [1]:
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path

import torch
import numpy as np
from tqdm import tqdm

try:
    from safetensors.torch import save_file
    from safetensors import safe_open
except ImportError:
    print("Please install safetensors: pip install safetensors")

In [2]:
def convert_pt_to_safetensors(input_dir: Path):
    """
    Converts all .pt files in a directory to .safetensors.
    Flattens the dict from [step][layer] -> "step.layer"
    """
    files = sorted(input_dir.glob("*.pt"))
    files = [f for f in files if f.name != "config.pt"]
    
    if not files:
        print(f"No .pt files found in {input_dir}")
        return

    for fp in tqdm(files, desc="Converting files"):
        # 1. Load the heavy .pt file
        payload = torch.load(fp, map_location="cpu", weights_only=True)
        gradients = payload["gradients"]
        metadata = payload.get("metadata", {})
        
        # 2. Flatten the nested structure for safetensors
        # key format: "step_idx.layer_name" (e.g., "1.down/31")
        flat_dict = {}
        for step_idx, layer_dict in gradients.items():
            for layer_name, tensor in layer_dict.items():
                # Ensure tensor is contiguous for safetensors
                key = f"{step_idx}.{layer_name}"
                flat_dict[key] = tensor.contiguous()
        
        # 3. Create output path
        out_path = fp.with_suffix(".safetensors")
        
        # 4. Save metadata as a JSON string in the header
        # safetensors metadata must be a dict of strings
        header_metadata = {"payload_metadata": json.dumps(metadata)}
        
        save_file(flat_dict, out_path, metadata=header_metadata)
        
        # Optional: Free memory
        del payload, gradients, flat_dict

In [3]:
# grad_dir = Path("outputs/grads")
# model = "llama-3.1-8b"
# # subset = "hand-crafted"
# subset = "algorithm-generated"

# convert_pt_to_safetensors(grad_dir / model / subset)

In [5]:
@dataclass
class StepIndex:
    """Row-level metadata for one entry in the stacked gradient matrix."""
    row:        int     # row index in G
    traj_idx:   int     # 1-based index into the loaded data list
    step_idx:   int     # step index within the trajectory
    role:       str     # e.g. "WebSurfer", "Orchestrator (thought)"
    is_mistake: bool    # whether this is the gold mistake step


@dataclass
class GradientStore:
    """
    Stores gradients for multiple weight names across the same set of steps.
    """
    # Maps: weight_name -> (T, d) matrix
    Gs: dict[str, torch.Tensor] 
    
    # Metadata is shared across all layers because step indices are identical
    index:       list[StepIndex]
    lookup:      dict[tuple[int, int], int]
    traj_meta:   dict[dict]
    traj_ranges: list[tuple[int, int]]
    device:      torch.device

    def __getitem__(self, weight_name: str) -> torch.Tensor:
        return self.Gs[weight_name]

    @property
    def layer_names(self) -> list[str]:
        return list(self.Gs.keys())


def get_all_weight_names(fp: Path):
    with safe_open(fp, framework="pt") as f:
        return sorted({k.split(".", 1)[1] for k in f.keys()})
        
def load_and_stack(
    model: str,
    subset: str,
    weight_names: list[str],  # List of layers to load, e.g. ["down/31", "up/31"]
    data_dir: Path,
    device: torch.device,
    grad_dir: Path,
):
    input_dir = grad_dir / model / subset
    files = sorted(input_dir.glob("*.safetensors"), key=lambda x: int(x.stem))
    
    # Initialize containers for each requested layer
    if weight_names == "all":
        weight_names = get_all_weight_names(files[0])
        
    layer_collections = {name: [] for name in weight_names}
    index: list[StepIndex] = []
    lookup: dict[tuple[int, int], int] = {}
    traj_meta: dict[dict] = {}
    traj_ranges: list[tuple[int, int]] = []

    row = 0
    for file_idx, fp in enumerate(tqdm(files, desc="Loading Multi-Layers")):
        traj_idx = int(fp.stem) # /path/to/1.safetensors -> 1
        
        with safe_open(fp, framework="pt", device="cpu") as f:
            # Load metadata
            header = f.metadata()
            metadata = json.loads(header.get("payload_metadata", "{}"))
            mistake_step = int(metadata.get("mistake_step", -1))
            traj_meta[traj_idx] = metadata
            
            # Use the first requested layer to determine step indices 
            # (assuming all layers exist for all steps)
            first_layer = weight_names[0]
            step_keys = [k for k in f.keys() if k.endswith(f".{first_layer}")]
            step_indices = sorted([int(k.split(".")[0]) for k in step_keys])
            
            # Load matching JSON for roles
            with open(data_dir / fp.with_suffix(".json").name) as jf:
                traj_data = json.load(jf)
                history = traj_data['history']
                
            start_row = row
            for step_idx in step_indices:
                # 1. Collect tensors for EVERY requested layer at this step
                for name in weight_names:
                    key = f"{step_idx}.{name}"
                    layer_collections[name].append(f.get_tensor(key))
                
                # 2. Record index (only once per step)
                index.append(StepIndex(
                    row=row, traj_idx=traj_idx, step_idx=step_idx,
                    role=history[step_idx]["role"],
                    is_mistake=(step_idx == mistake_step),
                ))
                lookup[(traj_idx, step_idx)] = row
                row += 1

            traj_ranges.append((start_row, row))

    # Convert lists to stacked matrices and move to device
    Gs = {
        name: torch.stack(tensors).to(device) 
        for name, tensors in layer_collections.items()
    }

    return GradientStore(
        Gs=Gs, index=index, lookup=lookup,
        traj_meta=traj_meta, traj_ranges=traj_ranges,
        device=device,
    )

In [6]:
device        = torch.device("cuda:2")
grad_dir      = Path("outputs/grads")
model_name    = "llama-3.1-8b"
subset        = "hand-crafted"
# subset        = "algorithm-generated"
data_dir  = Path("ww") / subset
weight_name   = "v/31"

print(f"Subset:       {subset}")
print(f"Model:        {model_name}")
print(f"Device:       {device}")
print(f"Config:       {weight_name}")

store = load_and_stack(
    model=model_name,
    subset=subset,
    weight_names="all",
    data_dir=data_dir,
    device=device,
    grad_dir=grad_dir
) 

Subset:       hand-crafted
Model:        llama-3.1-8b
Device:       cuda:2
Config:       v/31


Loading Multi-Layers: 100%|██████████| 58/58 [00:45<00:00,  1.28it/s]


In [8]:
import pandas as pd
import numpy as np
from typing import Callable
from scipy.stats import rankdata
from concurrent.futures import ThreadPoolExecutor
import torch

def standardize_role(role: str) -> str:
    if "orchestrator" in role.lower(): return "Orchestrator"
    else: return role

def compute_metrics(
    scores: np.ndarray,
    store: GradientStore,
    mistake_indices: list[int | None],  # absolute step_idx in history
    mistake_roles: list[str | None],
    ks: list[int],
    direction: str,
) -> dict:
    ascending    = (direction == "asc")
    total_trajs  = len(store.traj_ranges)
    step_hits    = {k: 0 for k in ks}
    agent_hits   = {k: 0 for k in ks}

    for (start, end), mistake_step, mistake_role in zip(
        store.traj_ranges, mistake_indices, mistake_roles
    ):
        if mistake_step is None:
            continue

        # Pair each entry with its score, then rank by score
        traj_entries = store.index[start:end]
        traj_scores  = scores[start:end]
        step_scores  = [(entry.step_idx, entry.role, score) 
                        for entry, score in zip(traj_entries, traj_scores)]
        step_scores.sort(key=lambda x: x[2], reverse=not ascending)

        ranked_steps  = [step_idx for step_idx, _, _ in step_scores]
        ranked_roles  = [standardize_role(role) for _, role, _ in step_scores]
        mistake_rank  = ranked_steps.index(mistake_step) + 1  # 1-based ranking.

        for k in ks:
            if mistake_rank <= k:
                step_hits[k] += 1
            if mistake_role in ranked_roles[:k]:
                agent_hits[k] += 1

    return {
        **{f"step@{k}_{direction}":  step_hits[k]  / total_trajs for k in ks},
        **{f"agent@{k}_{direction}": agent_hits[k] / total_trajs for k in ks},
    }

def evaluate_weights(
    store: GradientStore,
    scoring_fn: Callable[[torch.Tensor], torch.Tensor],
    ks: list[int] = [1, 3, 5],
) -> pd.DataFrame:

    # --- Phase 1: Compute scores for all weights ---
    all_scores: dict[str, np.ndarray] = {}
    for weight_name, G in tqdm(store.Gs.items(), desc="Scoring"):
        all_scores[weight_name] = scoring_fn(G).cpu().numpy()

    # --- Precompute trajectory metadata ---
    mistake_indices: list[int | None] = []
    mistake_roles:   list[str | None] = []

    for start, end in store.traj_ranges:
        traj_index  = store.index[start:end]
        mistake_entry = next((e for e in traj_index if e.is_mistake), None)
        mistake_role = store.traj_meta[mistake_entry.traj_idx]['mistake_agent']
        mistake_idx = mistake_entry.step_idx

        mistake_roles.append(mistake_role)
        mistake_indices.append(mistake_idx)
        # mistake_roles.append(mistake_entry.role if mistake_entry else None)

    # --- Phase 2: Evaluate predictions (parallelized over weights) ---
    results = []
    for weight_name, scores in tqdm(all_scores.items(), desc="Predicting"):
        row = {"weight": weight_name}
        for direction in ["asc", "desc"]:
            row |= compute_metrics(
                scores, 
                store, 
                mistake_indices, 
                mistake_roles, 
                ks, 
                direction
            )
        results.append(row)

    df = pd.DataFrame(results)
    df = df.sort_values("step@1_asc", ascending=False).reset_index(drop=True)
    return df

In [9]:
# Define scoring functions
l1_norm = lambda G: G.float().norm(p=1, dim=1)
l2_norm = lambda G: G.float().norm(p=2, dim=1)

# Run evaluation
df_l1 = evaluate_weights(store, scoring_fn=l1_norm, ks=[1, 3, 5])
df_l2 = evaluate_weights(store, scoring_fn=l2_norm, ks=[1, 3, 5])

Scoring:   0%|          | 0/291 [00:00<?, ?it/s]

Predicting: 100%|██████████| 291/291 [00:02<00:00, 145.00it/s]


In [12]:
df_l2

,weight,step@1_asc,step@3_asc,step@5_asc,agent@1_asc,agent@3_asc,agent@5_asc,step@1_desc,step@3_desc,step@5_desc,agent@1_desc,agent@3_desc,agent@5_desc
0,gate/11,0.241379,0.293103,0.517241,0.655172,0.758621,0.862069,0.000000,0.034483,0.120690,0.310345,0.310345,0.344828
1,down/30,0.241379,0.327586,0.413793,0.603448,0.775862,0.862069,0.017241,0.051724,0.103448,0.293103,0.344828,0.396552
2,q/12,0.241379,0.310345,0.413793,0.603448,0.741379,0.793103,0.000000,0.000000,0.120690,0.310345,0.310345,0.344828
3,gate/22,0.224138,0.344828,0.396552,0.620690,0.810345,0.862069,0.000000,0.017241,0.103448,0.310345,0.327586,0.379310
4,gate/12,0.224138,0.310345,0.448276,0.620690,0.741379,0.810345,0.000000,0.017241,0.120690,0.310345,0.310345,0.344828
...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,o/26,0.086207,0.241379,0.310345,0.500000,0.741379,0.793103,0.000000,0.051724,0.103448,0.310345,0.362069,0.396552
287,k/26,0.086207,0.293103,0.379310,0.500000,0.758621,0.879310,0.000000,0.017241,0.137931,0.293103,0.310345,0.362069
288,k/2,0.086207,0.293103,0.413793,0.568966,0.810345,0.896552,0.017241,0.051724,0.137931,0.310345,0.327586,0.362069
289,gate/1,0.086207,0.258621,0.362069,0.603448,0.827586,0.862069,0.017241,0.051724,0.137931,0.310345,0.327586,0.362069


In [25]:
ranks = evaluate_weights(store)

Evaluating Weights: 100%|██████████| 291/291 [00:02<00:00, 128.63it/s]


In [26]:
print(ranks.sort_values(["acc"], ascending=[False]).to_string())

           weight       acc       mrr
24        down/30  0.241379  0.339529
42        gate/17  0.224138  0.316107
114          k/24  0.224138  0.313016
254          up/5  0.224138  0.321986
251         up/30  0.224138  0.332639
73     in_norm/16  0.224138  0.326413
80     in_norm/22  0.224138  0.326009
79     in_norm/21  0.224138  0.332751
78     in_norm/20  0.224138  0.322542
37        gate/12  0.224138  0.329521
190   post_norm/5  0.224138  0.327443
232         up/13  0.224138  0.329843
234         up/15  0.224138  0.331291
171  post_norm/16  0.224138  0.327430
6         down/14  0.224138  0.334769
210          q/22  0.224138  0.331772
199          q/12  0.224138  0.322301
8         down/16  0.224138  0.322459
240         up/20  0.224138  0.322589
138          o/15  0.224138  0.340295
139          o/16  0.224138  0.331917
247         up/27  0.224138  0.302012
38        gate/13  0.206897  0.323029
21        down/28  0.206897  0.310537
40        gate/15  0.206897  0.314567
30         d